In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.dst_price_change_dataset AS

-- ============================================
-- PARAMETERS - UPDATE THESE
-- ============================================
WITH params AS (
  SELECT 
    DATE '2025-01-01' AS yoy_pre_start_date,
    DATE '2025-01-15' AS yoy_pre_end_date,
    DATE '2025-01-16' AS yoy_event_week_start,
    DATE '2025-01-30' AS yoy_post_end_date,
    DATE '2026-01-01' AS event_pre_start_date,
    DATE '2026-01-15' AS event_pre_end_date,
    DATE '2026-01-16' AS event_week_start_str,
    DATE '2026-01-30' AS event_post_end_date,
    ARRAY('Germany') AS countries,
    ARRAY() AS tour_ids
),

-- ============================================
-- Tour-level price observations
-- ============================================
tour_level_prices AS (
  SELECT 
    p.tour_id,
    dt.user_id AS supplier_id,
    dl.country_name,
    CASE 
      WHEN p.snapshot_date BETWEEN params.yoy_pre_start_date AND params.yoy_pre_end_date THEN 'PRE_LY'
      WHEN p.snapshot_date BETWEEN params.yoy_event_week_start AND params.yoy_post_end_date THEN 'POST_LY'
      WHEN p.snapshot_date BETWEEN params.event_pre_start_date AND params.event_pre_end_date THEN 'PRE_CY'
      WHEN p.snapshot_date BETWEEN params.event_week_start_str AND params.event_post_end_date THEN 'POST_CY'
    END AS period_window,
    CASE 
      WHEN p.snapshot_date BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date 
        THEN params.yoy_event_week_start
      ELSE params.event_week_start_str
    END AS event_start,
    p.travel_date,
    AVG(p.tour_from_price_red_per_adult) AS tour_from_price_red_per_adult,
    AVG(p.tour_from_price_black_per_adult) AS tour_from_price_black_per_adult,
    AVG(p.final_price_red_per_adult) AS final_price_red_per_adult,
    AVG(p.final_price_black_per_adult) AS final_price_black_per_adult,
    AVG(COALESCE(SIZE(p.availability_timeslots), 0)) AS timeslot_count
  FROM production.dwh.daily_tour_price_snapshot p
  INNER JOIN production.dwh.dim_tour dt ON p.tour_id = dt.tour_id
  INNER JOIN production.dwh.dim_location dl ON dt.location_id = dl.location_id
  CROSS JOIN params
  WHERE (
    (p.snapshot_date BETWEEN params.yoy_pre_start_date AND params.yoy_post_end_date
     AND p.travel_date BETWEEN params.yoy_event_week_start AND ADD_MONTHS(params.yoy_event_week_start, 12))
    OR
    (p.snapshot_date BETWEEN params.event_pre_start_date AND params.event_post_end_date
     AND p.travel_date BETWEEN params.event_week_start_str AND ADD_MONTHS(params.event_week_start_str, 12))
  )
    AND (SIZE(params.countries) = 0 OR ARRAY_CONTAINS(params.countries, dl.country_name))
    AND (SIZE(params.tour_ids) = 0 OR ARRAY_CONTAINS(params.tour_ids, dt.tour_id))
  GROUP BY 
    p.tour_id,
    dt.user_id,
    dl.country_name, 
    period_window, 
    event_start, 
    p.travel_date
),

-- ============================================
-- Tour-level averages per period per horizon
-- ============================================
tour_window_avg AS (
  SELECT
    tour_id,
    supplier_id,
    country_name,
    period_window,
    AVG(CASE WHEN travel_date BETWEEN event_start AND ADD_MONTHS(event_start, 3) 
             THEN final_price_red_per_adult END) AS avg_red_3m,
    AVG(CASE WHEN travel_date BETWEEN event_start AND ADD_MONTHS(event_start, 6) 
             THEN final_price_red_per_adult END) AS avg_red_6m,
    AVG(CASE WHEN travel_date BETWEEN event_start AND ADD_MONTHS(event_start, 12) 
             THEN final_price_red_per_adult END) AS avg_red_12m
  FROM tour_level_prices
  WHERE period_window IS NOT NULL
  GROUP BY tour_id, supplier_id, country_name, period_window
),

-- ============================================
-- Balanced panel: tour must exist in all 4 windows
-- ============================================
tour_pivot AS (
  SELECT
    tour_id,
    supplier_id,
    country_name,
    MAX(CASE WHEN period_window = 'PRE_LY'  THEN avg_red_3m END)  AS pre_ly_3m,
    MAX(CASE WHEN period_window = 'POST_LY' THEN avg_red_3m END)  AS post_ly_3m,
    MAX(CASE WHEN period_window = 'PRE_CY'  THEN avg_red_3m END)  AS pre_cy_3m,
    MAX(CASE WHEN period_window = 'POST_CY' THEN avg_red_3m END)  AS post_cy_3m,
    MAX(CASE WHEN period_window = 'PRE_LY'  THEN avg_red_6m END)  AS pre_ly_6m,
    MAX(CASE WHEN period_window = 'POST_LY' THEN avg_red_6m END)  AS post_ly_6m,
    MAX(CASE WHEN period_window = 'PRE_CY'  THEN avg_red_6m END)  AS pre_cy_6m,
    MAX(CASE WHEN period_window = 'POST_CY' THEN avg_red_6m END)  AS post_cy_6m,
    MAX(CASE WHEN period_window = 'PRE_LY'  THEN avg_red_12m END) AS pre_ly_12m,
    MAX(CASE WHEN period_window = 'POST_LY' THEN avg_red_12m END) AS post_ly_12m,
    MAX(CASE WHEN period_window = 'PRE_CY'  THEN avg_red_12m END) AS pre_cy_12m,
    MAX(CASE WHEN period_window = 'POST_CY' THEN avg_red_12m END) AS post_cy_12m
  FROM tour_window_avg
  GROUP BY tour_id, supplier_id, country_name
  HAVING
    MAX(CASE WHEN period_window = 'PRE_LY'  THEN 1 END) = 1
    AND MAX(CASE WHEN period_window = 'POST_LY' THEN 1 END) = 1
    AND MAX(CASE WHEN period_window = 'PRE_CY'  THEN 1 END) = 1
    AND MAX(CASE WHEN period_window = 'POST_CY' THEN 1 END) = 1
),

-- ============================================
-- DID per tour on absolute price levels
-- ============================================
tour_did AS (
  SELECT
    tour_id,
    supplier_id,
    country_name,
    (post_ly_3m  - pre_ly_3m)  / NULLIF(pre_ly_3m, 0)  AS ly_pct_3m,
    (post_ly_6m  - pre_ly_6m)  / NULLIF(pre_ly_6m, 0)  AS ly_pct_6m,
    (post_ly_12m - pre_ly_12m) / NULLIF(pre_ly_12m, 0)  AS ly_pct_12m,
    ((post_cy_3m  - pre_cy_3m)  - (post_ly_3m  - pre_ly_3m))  / NULLIF(pre_cy_3m, 0)  AS did_pct_3m,
    ((post_cy_6m  - pre_cy_6m)  - (post_ly_6m  - pre_ly_6m))  / NULLIF(pre_cy_6m, 0)  AS did_pct_6m,
    ((post_cy_12m - pre_cy_12m) - (post_ly_12m - pre_ly_12m)) / NULLIF(pre_cy_12m, 0) AS did_pct_12m
  FROM tour_pivot
),

-- ============================================
-- Change flags: LY = raw organic, CY = DID-corrected
-- ============================================
tour_changes AS (
  SELECT
    'LY' AS year,
    country_name,
    tour_id,
    supplier_id,
    ly_pct_3m  AS pct_change_3m,
    ly_pct_6m  AS pct_change_6m,
    ly_pct_12m AS pct_change_12m,
    CASE WHEN ABS(COALESCE(ly_pct_3m, 0))  > 0.01 THEN 1 ELSE 0 END AS tour_changed_3m,
    CASE WHEN ABS(COALESCE(ly_pct_6m, 0))  > 0.01 THEN 1 ELSE 0 END AS tour_changed_6m,
    CASE WHEN ABS(COALESCE(ly_pct_12m, 0)) > 0.01 THEN 1 ELSE 0 END AS tour_changed_12m
  FROM tour_did

  UNION ALL

  SELECT
    'CY' AS year,
    country_name,
    tour_id,
    supplier_id,
    did_pct_3m  AS pct_change_3m,
    did_pct_6m  AS pct_change_6m,
    did_pct_12m AS pct_change_12m,
    CASE WHEN ABS(COALESCE(did_pct_3m, 0))  > 0.01 THEN 1 ELSE 0 END AS tour_changed_3m,
    CASE WHEN ABS(COALESCE(did_pct_6m, 0))  > 0.01 THEN 1 ELSE 0 END AS tour_changed_6m,
    CASE WHEN ABS(COALESCE(did_pct_12m, 0)) > 0.01 THEN 1 ELSE 0 END AS tour_changed_12m
  FROM tour_did
),

-- ============================================
-- Summary: share of suppliers/tours that changed
-- ============================================
change_summary AS (
  SELECT
    year,
    country_name,
    COUNT(DISTINCT supplier_id) AS total_suppliers_in_change,
    COUNT(DISTINCT CASE WHEN tour_changed_3m = 1 THEN supplier_id END) AS suppliers_changed_3m,
    COUNT(DISTINCT CASE WHEN tour_changed_6m = 1 THEN supplier_id END) AS suppliers_changed_6m,
    COUNT(DISTINCT CASE WHEN tour_changed_12m = 1 THEN supplier_id END) AS suppliers_changed_12m,
    COUNT(DISTINCT tour_id) AS total_tours_in_change,
    SUM(tour_changed_3m) AS tours_changed_3m,
    SUM(tour_changed_6m) AS tours_changed_6m,
    SUM(tour_changed_12m) AS tours_changed_12m,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN tour_changed_3m = 1 THEN supplier_id END) / COUNT(DISTINCT supplier_id), 2) AS pct_suppliers_changed_3m,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN tour_changed_6m = 1 THEN supplier_id END) / COUNT(DISTINCT supplier_id), 2) AS pct_suppliers_changed_6m,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN tour_changed_12m = 1 THEN supplier_id END) / COUNT(DISTINCT supplier_id), 2) AS pct_suppliers_changed_12m,
    ROUND(100.0 * SUM(tour_changed_3m) / COUNT(DISTINCT tour_id), 2) AS pct_tours_changed_3m,
    ROUND(100.0 * SUM(tour_changed_6m) / COUNT(DISTINCT tour_id), 2) AS pct_tours_changed_6m,
    ROUND(100.0 * SUM(tour_changed_12m) / COUNT(DISTINCT tour_id), 2) AS pct_tours_changed_12m
  FROM tour_changes
  GROUP BY year, country_name
),

-- ============================================
-- NEW: Balanced panel medians (same tours, equal weight)
-- Each tour gets one average, then we take the median across tours
-- ============================================
balanced_tour_prices AS (
  SELECT
    tlp.tour_id,
    tlp.country_name,
    tlp.period_window,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
             THEN tlp.final_price_red_per_adult END) AS tour_avg_price_3m,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
             THEN tlp.final_price_red_per_adult END) AS tour_avg_price_6m,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
             THEN tlp.final_price_red_per_adult END) AS tour_avg_price_12m,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
             THEN tlp.timeslot_count END) AS tour_avg_timeslots_3m,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
             THEN tlp.timeslot_count END) AS tour_avg_timeslots_6m,
    AVG(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
             THEN tlp.timeslot_count END) AS tour_avg_timeslots_12m
  FROM tour_level_prices tlp
  INNER JOIN tour_pivot tp ON tlp.tour_id = tp.tour_id
  WHERE tlp.period_window IS NOT NULL
  GROUP BY tlp.tour_id, tlp.country_name, tlp.period_window
),

balanced_medians AS (
  SELECT
    country_name,
    period_window,
    PERCENTILE_APPROX(tour_avg_price_3m, 0.5)      AS balanced_median_price_3m,
    PERCENTILE_APPROX(tour_avg_price_6m, 0.5)      AS balanced_median_price_6m,
    PERCENTILE_APPROX(tour_avg_price_12m, 0.5)     AS balanced_median_price_12m,
    PERCENTILE_APPROX(tour_avg_timeslots_3m, 0.5)  AS balanced_median_timeslots_3m,
    PERCENTILE_APPROX(tour_avg_timeslots_6m, 0.5)  AS balanced_median_timeslots_6m,
    PERCENTILE_APPROX(tour_avg_timeslots_12m, 0.5) AS balanced_median_timeslots_12m
  FROM balanced_tour_prices
  GROUP BY country_name, period_window
)

-- ============================================
-- Final output
-- ============================================
SELECT 
  tlp.country_name,
  tlp.period_window,
  COUNT(DISTINCT tlp.supplier_id) AS total_suppliers,
  COUNT(DISTINCT tlp.tour_id) AS total_tours,

  -- Change counts and percentages
  cs.suppliers_changed_3m,
  cs.suppliers_changed_6m,
  cs.suppliers_changed_12m,
  cs.pct_suppliers_changed_3m,
  cs.pct_suppliers_changed_6m,
  cs.pct_suppliers_changed_12m,
  cs.tours_changed_3m,
  cs.tours_changed_6m,
  cs.tours_changed_12m,
  cs.pct_tours_changed_3m,
  cs.pct_tours_changed_6m,
  cs.pct_tours_changed_12m,

  -- Unbalanced medians (all tours, observation-weighted)
  PERCENTILE_APPROX(tlp.tour_from_price_red_per_adult, 0.5) AS median_from_red_price,
  STDDEV(tlp.tour_from_price_red_per_adult) AS stddev_from_red_price,
  PERCENTILE_APPROX(tlp.tour_from_price_black_per_adult, 0.5) AS median_from_black_price,
  STDDEV(tlp.tour_from_price_black_per_adult) AS stddev_from_black_price,

  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
           THEN tlp.final_price_red_per_adult END, 0.5) AS median_final_red_3m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
              THEN tlp.final_price_red_per_adult END) AS stddev_final_red_3m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
           THEN tlp.final_price_red_per_adult END, 0.5) AS median_final_red_6m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
              THEN tlp.final_price_red_per_adult END) AS stddev_final_red_6m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
           THEN tlp.final_price_red_per_adult END, 0.5) AS median_final_red_12m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
              THEN tlp.final_price_red_per_adult END) AS stddev_final_red_12m,

  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
           THEN tlp.final_price_black_per_adult END, 0.5) AS median_final_black_3m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
              THEN tlp.final_price_black_per_adult END) AS stddev_final_black_3m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
           THEN tlp.final_price_black_per_adult END, 0.5) AS median_final_black_6m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
              THEN tlp.final_price_black_per_adult END) AS stddev_final_black_6m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
           THEN tlp.final_price_black_per_adult END, 0.5) AS median_final_black_12m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
              THEN tlp.final_price_black_per_adult END) AS stddev_final_black_12m,

  -- Unbalanced timeslot medians (all tours)
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
           THEN tlp.timeslot_count END, 0.5) AS median_timeslots_3m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 3) 
              THEN tlp.timeslot_count END) AS stddev_timeslots_3m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
           THEN tlp.timeslot_count END, 0.5) AS median_timeslots_6m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 6) 
              THEN tlp.timeslot_count END) AS stddev_timeslots_6m,
  PERCENTILE_APPROX(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
           THEN tlp.timeslot_count END, 0.5) AS median_timeslots_12m,
  STDDEV(CASE WHEN tlp.travel_date BETWEEN tlp.event_start AND ADD_MONTHS(tlp.event_start, 12) 
              THEN tlp.timeslot_count END) AS stddev_timeslots_12m,

  -- NEW: Balanced panel medians (same tours, equal weight per tour)
  bm.balanced_median_price_3m,
  bm.balanced_median_price_6m,
  bm.balanced_median_price_12m,
  bm.balanced_median_timeslots_3m,
  bm.balanced_median_timeslots_6m,
  bm.balanced_median_timeslots_12m

FROM tour_level_prices tlp
LEFT JOIN change_summary cs 
  ON tlp.country_name = cs.country_name
  AND CASE 
    WHEN tlp.period_window IN ('PRE_LY', 'POST_LY') THEN 'LY' 
    ELSE 'CY' 
  END = cs.year
LEFT JOIN balanced_medians bm
  ON tlp.country_name = bm.country_name
  AND tlp.period_window = bm.period_window
WHERE tlp.period_window IS NOT NULL
GROUP BY 
  tlp.country_name, 
  tlp.period_window,
  cs.suppliers_changed_3m,
  cs.suppliers_changed_6m,
  cs.suppliers_changed_12m,
  cs.pct_suppliers_changed_3m,
  cs.pct_suppliers_changed_6m,
  cs.pct_suppliers_changed_12m,
  cs.tours_changed_3m,
  cs.tours_changed_6m,
  cs.tours_changed_12m,
  cs.pct_tours_changed_3m,
  cs.pct_tours_changed_6m,
  cs.pct_tours_changed_12m,
  bm.balanced_median_price_3m,
  bm.balanced_median_price_6m,
  bm.balanced_median_price_12m,
  bm.balanced_median_timeslots_3m,
  bm.balanced_median_timeslots_6m,
  bm.balanced_median_timeslots_12m;
